# Example: Performance Comparison of QR Iteration Implementation
In this example, we compare the performance of our QR iteration algorithm implementation against Julia's optimized built-in eigendecomposition functions. This comparison illustrates the practical trade-offs between custom implementations and established numerical libraries.

> __Learning Objectives:__
> 
> By the end of this example, you should be able to:
> 
> * __Validate custom eigendecomposition implementations:__ Compare eigenvalues and eigenvectors computed by custom QR iteration against standard library functions using vector norms and tolerance-based comparisons.
> * __Benchmark computational performance:__ Use Julia's benchmarking tools to measure execution time, memory usage, and allocation patterns for competing implementations.
> * __Evaluate implementation trade-offs:__ Assess when custom implementations are justified versus when established libraries should be preferred based on performance metrics and development cost.

Let's get started!

___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.
___

## Task 1: Compute Eigendecomposition Using Both Methods
In this task, we compute the eigenvalues and eigenvectors of a test matrix using both our custom QR iteration implementation and Julia's built-in function. 

> __Test matrix:__ We use a $3 \times 3$ matrix with dominant diagonal entries (3.0, 7.0, 10.0) and small off-diagonal perturbations, making it well-conditioned for eigendecomposition:
> 
> $$
> \mathbf{A} = \begin{bmatrix}
> 3.0 & -0.3 & -0.2 \\
> 0.1 & 7.0 & -0.3 \\
> 0.3 & -0.2 & 10.0 \\
> \end{bmatrix}
> $$

Let's define this test matrix:

In [2]:
# Setup matrix the n x n matrix A (n = 3)
A = [3.0 -0.3 -0.2 ; 0.1 7.0 -0.3 ; 0.3 -0.2 10.0]

3×3 Matrix{Float64}:
 3.0  -0.3  -0.2
 0.1   7.0  -0.3
 0.3  -0.2  10.0

### Built-in eigendecomposition
First, let's compute the eigendecomposition using [Julia's built-in `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen).

> __Eigen factorization:__ The `eigen(...)` function returns an `Eigen` factorization object containing both eigenvalues (accessed via `.values`) and eigenvectors (accessed via `.vectors`, where each column is an eigenvector).

Let's compute the eigendecomposition and extract the results:

In [3]:
# Decompose using the built-in function
F = eigen(A);   # eigenvalues and vectors in F of type Eigen
λ = F.values;   # vector of eigenvalues
V = F.vectors;  # 3 x 3 matrix of eigenvectors, each col is an eigenvector

In [4]:
# Display eigenvalues and eigenvectors
println("Eigenvalues λ:")
display(λ)
println("\nEigenvector matrix V (each column is an eigenvector):")
display(V)

3-element Vector{Float64}:
  3.017277143754452
  6.9699146114784165
 10.012808244767134

Eigenvalues λ:

Eigenvector matrix V (each column is an eigenvector):


3×3 Matrix{Float64}:
 -0.998641    0.0788277  -0.024097
  0.0283674  -0.994181   -0.099848
  0.0437173  -0.0734251   0.994711

### Custom QR iteration
Now, let's compute the same eigendecomposition using [our `qriteration(...)` function](src/Compute.jl).

> __Convergence parameters:__ We specify `maxiter=10000` as the maximum number of QR iterations and `tolerance=1e-6` as the convergence threshold. The algorithm terminates when off-diagonal elements become sufficiently small, indicating convergence to diagonal form.

Let's compute the eigendecomposition using our implementation:

In [5]:
# Call our qriteration function
(λ̂, V̂_dict) = qriteration(A; maxiter=10000, tolerance=1e-6);

# Convert eigenvector dictionary to matrix format for comparison
n = length(λ̂);
V̂ = zeros(n, n);
for i ∈ 1:n
    V̂[:,i] = V̂_dict[i];
end

___

## Task 2: Validate Implementation Correctness
In this task, we verify that our custom QR iteration produces results consistent with the built-in function. We'll compare eigenvalues and eigenvectors using vector norms.

> __Vector norms for comparison:__ A [norm](https://en.wikipedia.org/wiki/Norm_(mathematics)) quantifies the magnitude of a vector or the difference between two vectors. The 2-norm (Euclidean norm) of a vector $\mathbf{x} \in \mathbb{R}^{n}$ is $\|\mathbf{x}\|_{2} = \sqrt{\sum_{i=1}^{n} x_i^2}$. We use [Julia's `norm(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.norm) which defaults to the 2-norm for vectors.

Let's compare the computed eigenvalues and eigenvectors systematically.

### Eigenvalue comparison
Let's compare the eigenvalues computed by both methods and quantify their agreement:

In [6]:
# Compare eigenvalues side by side
println("Eigenvalue comparison [built-in | custom]:")
display([λ λ̂])

# Compute 2-norm of difference
diff_norm = norm(λ - λ̂, 2)
println("\n||λ - λ̂||₂ = $diff_norm")

# Check element-wise agreement
println("\nElement-wise agreement (tolerance 1e-5):")
display(isapprox.(λ̂, λ, atol=1e-5))

3×2 Matrix{Float64}:
  3.01728   3.01728
  6.96991   6.96991
 10.0128   10.0128

3-element BitVector:
 1
 1
 1

Eigenvalue comparison [built-in | custom]:

||λ - λ̂||₂ = 2.2554654274383627e-6

Element-wise agreement (tolerance 1e-5):


The eigenvalues match closely between both implementations, confirming correctness.

### Eigenvector comparison
Let's verify agreement for all eigenvectors:

> __Sign ambiguity:__ Eigenvectors are defined up to a scalar multiple. If $\mathbf{A}\mathbf{v} = \lambda\mathbf{v}$, then $\mathbf{A}(-\mathbf{v}) = \lambda(-\mathbf{v})$, so both $\mathbf{v}$ and $-\mathbf{v}$ are equally valid eigenvectors. Different algorithms may converge to opposite signs for the same eigenspace.

Let's check agreement accounting for sign ambiguity:

In [7]:
for i ∈ 1:length(λ)
    # Check both v and -v since eigenvectors are defined up to sign
    diff_norm_pos = norm(V[:,i] - V̂[:,i], 2)
    diff_norm_neg = norm(V[:,i] + V̂[:,i], 2)
    diff_norm = min(diff_norm_pos, diff_norm_neg)
    println("Eigenvector $i: min(||V[:,i] - V̂[:,i]||₂, ||V[:,i] + V̂[:,i]||₂) = $(diff_norm)")
end

Eigenvector 1: min(||V[:,i] - V̂[:,i]||₂, ||V[:,i] + V̂[:,i]||₂) = 2.1136330772906852e-13
Eigenvector 2: min(||V[:,i] - V̂[:,i]||₂, ||V[:,i] + V̂[:,i]||₂) = 4.897779441128817e-8
Eigenvector 3: min(||V[:,i] - V̂[:,i]||₂, ||V[:,i] + V̂[:,i]||₂) = 2.147857191398826e-8


All eigenvectors agree to within numerical tolerance when accounting for sign ambiguity, confirming implementation correctness.
___

## Task 3: Performance Benchmarking
In this task, we benchmark both implementations to compare execution time, memory usage, and allocation patterns. Having validated correctness, we now measure the computational cost.

> __Benchmarking methodology:__ We use [Julia's `BenchmarkTools.jl` package](https://github.com/JuliaCI/BenchmarkTools.jl) to measure performance. The `@benchmark` macro runs the code multiple times to obtain reliable statistics, accounting for compilation overhead and runtime variability.

Let's benchmark the built-in function first:

In [8]:
@benchmark eigen($A)

BenchmarkTools.Trial: 10000 samples with 10 evaluations per sample.
 Range (min … max):  1.821 μs …  1.158 ms  ┊ GC (min … max): 0.00% … 99.48%
 Time  (median):     1.992 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.205 μs ± 12.834 μs  ┊ GC (mean ± σ):  7.74% ±  1.40%

    ▂▄▆▇█████▇▇▇▇▆▆▅▄▄▃▁▁▁▁                                  ▃
  ▄██████████████████████████▇█▇██▇▇▆██▇▇▇▇█▇▇▇█▇▇███▇▇▇▆▆▆▇ █
  1.82 μs      Histogram: log(frequency) by time     2.77 μs <

 Memory estimate: 4.50 KiB, allocs estimate: 24.

Now let's benchmark our custom QR iteration:

In [9]:
@benchmark qriteration($A)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  12.166 μs …  4.838 ms  ┊ GC (min … max): 0.00% … 98.83%
 Time  (median):     13.000 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   13.819 μs ± 48.272 μs  ┊ GC (mean ± σ):  3.46% ±  0.99%

      ▁▁█▃▅                                                    
  ▁▁▃▃█████▇▆▄▃▄▃▃▂▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  12.2 μs         Histogram: frequency by time        18.2 μs <

 Memory estimate: 24.27 KiB, allocs estimate: 326.

The built-in function typically achieves execution times 2-4 times faster with 2-3 times less memory allocation compared to our custom implementation.

> __Performance gap explanation:__ This difference reflects years of optimization work by numerical computing experts, including BLAS/LAPACK integration, cache-efficient algorithms, and optimized memory management. While implementing algorithms from scratch provides learning value and algorithmic insight, production code should leverage established numerical libraries that have been extensively tested and optimized for performance.
___

## Summary
This example compared a custom QR iteration implementation against Julia's optimized eigendecomposition function, demonstrating that correctness alone does not guarantee practical performance.

> __Key Takeaways:__
>
> * **Custom implementations can match correctness of optimized libraries:** Our QR iteration produces eigenvalues and eigenvectors that agree with built-in functions to within numerical tolerance, validating the algorithmic correctness.
> * **Performance differences favor established libraries:** Benchmark results show that optimized library functions achieve significantly faster execution times and lower memory usage through years of refinement and low-level optimization.
> * **Implementation decisions involve trade-offs:** Custom implementations provide learning value and algorithmic insight, but production code should leverage well-tested numerical libraries to benefit from accumulated optimization effort and reliability.

This benchmark reinforces the principle that for numerical linear algebra, established libraries like LAPACK provide both correctness and performance that custom implementations rarely match.
___